In [55]:
import pandas as pd
import os
import re
import numpy as np

In [56]:
SCRIPT_DIR_PATH = os.getcwd()
CB_DIR_PATH = os.path.dirname(SCRIPT_DIR_PATH)
SSP_MODELING_DIR_PATH = os.path.dirname(CB_DIR_PATH)
TORNADO_DATA_DIR_PATH = os.path.join(SCRIPT_DIR_PATH, "data")
INPUT_DATA_DIR_PATH = os.path.join(TORNADO_DATA_DIR_PATH, "input")
OUTPUT_DATA_DIR_PATH = os.path.join(TORNADO_DATA_DIR_PATH, "output")

In [57]:
def add_sector_and_transformation_fields(df: pd.DataFrame, strategy_col: str = "strategy") -> pd.DataFrame:
    """
    Creates:
      - sector: sector code (e.g., AGRC)
      - transformation_name: transformation text after '... - <SECTOR>:'
    """
    df = df.copy()

    # --- sector extraction (captures 3-6 uppercase letters before colon) ---
    # Example: "Singleton - Default Value - AGRC: Improve rice..." -> AGRC
    df["sector"] = df[strategy_col].str.extract(r"-\s*([A-Z]{3,6})\s*:", expand=False)

    # Special case: baseline strategy
    df.loc[df[strategy_col].str.contains(r"^Strategy\s+TX:BASE", regex=True, na=False), "sector"] = "BASE"

    # --- transformation_name extraction ---
    # Keep only text after "<SECTOR>:"
    # Example -> "Improve rice..."
    df["transformation_name"] = df[strategy_col].str.extract(r":\s*(.*)$", expand=False)

    # If it's baseline, keep the full strategy string as the name (or label it as BASE)
    base_mask = df[strategy_col].str.contains(r"^Strategy\s+TX:BASE", regex=True, na=False)
    df.loc[base_mask, "transformation_name"] = "BASE"

    # Clean whitespace
    df["transformation_name"] = df["transformation_name"].fillna("").str.strip()

    return df

## Load and process emission data

In [58]:
# Load the decomposed emissions long format data
emissions_df = pd.read_csv(os.path.join(INPUT_DATA_DIR_PATH, "decomposed_emissions_libya_2023_sisepuede_results_sisepuede_run_2026-03-13T07;36;11.592365_WIDE_INPUTS_OUTPUTS.csv"))
emissions_df.head()

,strategy_id,primary_id,ID,sector,subsector,value,Year,Gas,design_id,future_id,strategy,Code,Contry,source
0,0.0,0.0,1.A.1 - Energy Industries:CH4,1 - Energy,1.A.1 - Energy Industries,0.050945,2023,CH4,0.0,0.0,Strategy TX:BASE,LBY,libya,SISEPUEDE
1,0.0,0.0,1.A.1 - Energy Industries:CH4,1 - Energy,1.A.1 - Energy Industries,0.028309,2024,CH4,0.0,0.0,Strategy TX:BASE,LBY,libya,SISEPUEDE
2,0.0,0.0,1.A.1 - Energy Industries:CH4,1 - Energy,1.A.1 - Energy Industries,0.028522,2025,CH4,0.0,0.0,Strategy TX:BASE,LBY,libya,SISEPUEDE
3,0.0,0.0,1.A.1 - Energy Industries:CH4,1 - Energy,1.A.1 - Energy Industries,0.028730,2026,CH4,0.0,0.0,Strategy TX:BASE,LBY,libya,SISEPUEDE
4,0.0,0.0,1.A.1 - Energy Industries:CH4,1 - Energy,1.A.1 - Energy Industries,0.028938,2027,CH4,0.0,0.0,Strategy TX:BASE,LBY,libya,SISEPUEDE


In [59]:
print(emissions_df.primary_id.nunique())

4


In [60]:
print(emissions_df.primary_id.nunique())

4


In [61]:
# check unique strategy
emissions_df['strategy'].unique()

array(['Strategy TX:BASE', 'NDC', 'LEP', 'Conditional', 'Historical'],
      dtype=object)

In [62]:
# Drop historical and tx:base from df
filtered_emissions_df = emissions_df.loc[~emissions_df['strategy'].isin(['Historical', 'Strategy TX:BASE'])]
print(emissions_df['strategy'].nunique())
print(filtered_emissions_df['strategy'].nunique())

5
3


In [63]:
filtered_emissions_df["strategy"].unique()

array(['NDC', 'LEP', 'Conditional'], dtype=object)

In [64]:
filtered_emissions_df.tail()

,strategy_id,primary_id,ID,sector,subsector,value,Year,Gas,design_id,future_id,strategy,Code,Contry,source
4587,6005.0,76076.0,5- CCSQ:N2O,5- CCSQ,5- CCSQ,0.0,2046,N2O,0.0,0.0,Conditional,LBY,libya,SISEPUEDE
4588,6005.0,76076.0,5- CCSQ:N2O,5- CCSQ,5- CCSQ,0.0,2047,N2O,0.0,0.0,Conditional,LBY,libya,SISEPUEDE
4589,6005.0,76076.0,5- CCSQ:N2O,5- CCSQ,5- CCSQ,0.0,2048,N2O,0.0,0.0,Conditional,LBY,libya,SISEPUEDE
4590,6005.0,76076.0,5- CCSQ:N2O,5- CCSQ,5- CCSQ,0.0,2049,N2O,0.0,0.0,Conditional,LBY,libya,SISEPUEDE
4591,6005.0,76076.0,5- CCSQ:N2O,5- CCSQ,5- CCSQ,0.0,2050,N2O,0.0,0.0,Conditional,LBY,libya,SISEPUEDE


In [65]:
# Now concat the original base df and the filtered emissions df
tornado_emissions_df = filtered_emissions_df
tornado_emissions_df['strategy'].unique()

array(['NDC', 'LEP', 'Conditional'], dtype=object)

In [66]:
tornado_emissions_df.head()

,strategy_id,primary_id,ID,sector,subsector,value,Year,Gas,design_id,future_id,strategy,Code,Contry,source
1148,6003.0,74074.0,1.A.1 - Energy Industries:CH4,1 - Energy,1.A.1 - Energy Industries,0.050945,2023,CH4,0.0,0.0,NDC,LBY,libya,SISEPUEDE
1149,6003.0,74074.0,1.A.1 - Energy Industries:CH4,1 - Energy,1.A.1 - Energy Industries,0.028309,2024,CH4,0.0,0.0,NDC,LBY,libya,SISEPUEDE
1150,6003.0,74074.0,1.A.1 - Energy Industries:CH4,1 - Energy,1.A.1 - Energy Industries,0.028522,2025,CH4,0.0,0.0,NDC,LBY,libya,SISEPUEDE
1151,6003.0,74074.0,1.A.1 - Energy Industries:CH4,1 - Energy,1.A.1 - Energy Industries,0.028730,2026,CH4,0.0,0.0,NDC,LBY,libya,SISEPUEDE
1152,6003.0,74074.0,1.A.1 - Energy Industries:CH4,1 - Energy,1.A.1 - Energy Industries,0.028938,2027,CH4,0.0,0.0,NDC,LBY,libya,SISEPUEDE


In [67]:
tornado_emissions_df.loc[
    tornado_emissions_df["subsector"].str.startswith("2."), "subsector"
] = "2 - IPPU"

In [68]:
# Aggregate by strategy_id, primary_id and strategy, and sum value
tornado_emissions_agg_df = tornado_emissions_df.groupby(
    ['strategy_id', 'primary_id', 'strategy', 'sector', 'subsector']
)['value'].sum().reset_index()

tornado_emissions_agg_df.head()


,strategy_id,primary_id,strategy,sector,subsector,value
0,6003.0,74074.0,NDC,1 - Energy,1.A.1 - Energy Industries,960.395602
1,6003.0,74074.0,NDC,1 - Energy,1.A.2 - Manufacturing Industries and Construction,51.611824
2,6003.0,74074.0,NDC,1 - Energy,1.A.3 - Transport,873.965289
3,6003.0,74074.0,NDC,1 - Energy,1.A.4 - Other Sectors,31.779867
4,6003.0,74074.0,NDC,1 - Energy,1.B - Fugitive emissions from fuels,793.721312


In [69]:
tornado_emissions_agg_df.tail()

,strategy_id,primary_id,strategy,sector,subsector,value
28,6005.0,76076.0,Conditional,"3 - Agriculture, Forestry, and Other Land Use",3.A - Livestock,32.285596
29,6005.0,76076.0,Conditional,"3 - Agriculture, Forestry, and Other Land Use",3.C - Aggregate sources and non-CO2 emissions ...,22.590322
30,6005.0,76076.0,Conditional,4 - Waste,4.A - Solid Waste Disposal,14.548662
31,6005.0,76076.0,Conditional,4 - Waste,4.D - Wastewater Treatment and Discharge,12.974991
32,6005.0,76076.0,Conditional,5- CCSQ,5- CCSQ,0.000000


In [70]:
tornado_emissions_agg_df['subsector'].unique()

array(['1.A.1 - Energy Industries',
       '1.A.2 - Manufacturing Industries and Construction',
       '1.A.3 - Transport', '1.A.4 - Other Sectors',
       '1.B - Fugitive emissions from fuels', '2 - IPPU',
       '3.A - Livestock',
       '3.C - Aggregate sources and non-CO2 emissions sources on land',
       '4.A - Solid Waste Disposal',
       '4.D - Wastewater Treatment and Discharge', '5- CCSQ'],
      dtype=object)

In [71]:
# check if strategy id nunique matches amount of rows
print(tornado_emissions_agg_df['strategy_id'].nunique())
print(tornado_emissions_agg_df.shape[0])

3
33


In [72]:
# rename value to emission_total
tornado_emissions_agg_df = tornado_emissions_agg_df.rename(columns={'value': 'emission_total'})

# create base_emission_total per subsector using strategy_id == 6003
base_by_subsector = tornado_emissions_agg_df.loc[
    tornado_emissions_agg_df['strategy_id'] == 6003, ['subsector', 'emission_total']
].rename(columns={'emission_total': 'base_emission_total'})

tornado_emissions_agg_df = tornado_emissions_agg_df.merge(base_by_subsector, on='subsector', how='left')

# calculate emission difference column
tornado_emissions_agg_df['emission_diff'] = tornado_emissions_agg_df['emission_total'] - tornado_emissions_agg_df['base_emission_total']

tornado_emissions_agg_df['emission_diff'] = tornado_emissions_agg_df['emission_diff'].round(1)

tornado_emissions_agg_df.head(40)

,strategy_id,primary_id,strategy,sector,subsector,emission_total,base_emission_total,emission_diff
0,6003.0,74074.0,NDC,1 - Energy,1.A.1 - Energy Industries,960.395602,960.395602,0.0
1,6003.0,74074.0,NDC,1 - Energy,1.A.2 - Manufacturing Industries and Construction,51.611824,51.611824,0.0
2,6003.0,74074.0,NDC,1 - Energy,1.A.3 - Transport,873.965289,873.965289,0.0
3,6003.0,74074.0,NDC,1 - Energy,1.A.4 - Other Sectors,31.779867,31.779867,0.0
4,6003.0,74074.0,NDC,1 - Energy,1.B - Fugitive emissions from fuels,793.721312,793.721312,0.0
5,6003.0,74074.0,NDC,2 - Industrial Processes and Product Use,2 - IPPU,122.231498,122.231498,0.0
6,6003.0,74074.0,NDC,"3 - Agriculture, Forestry, and Other Land Use",3.A - Livestock,32.904753,32.904753,0.0
7,6003.0,74074.0,NDC,"3 - Agriculture, Forestry, and Other Land Use",3.C - Aggregate sources and non-CO2 emissions ...,23.129455,23.129455,0.0
8,6003.0,74074.0,NDC,4 - Waste,4.A - Solid Waste Disposal,14.689550,14.689550,0.0
9,6003.0,74074.0,NDC,4 - Waste,4.D - Wastewater Treatment and Discharge,13.686307,13.686307,0.0


In [73]:
tornado_emissions_agg_df['strategy'].unique()

array(['NDC', 'LEP', 'Conditional'], dtype=object)

In [74]:
tornado_emissions_agg_df = tornado_emissions_agg_df[tornado_emissions_agg_df['strategy']!='NDC'] 

In [75]:
tornado_emissions_agg_df['strategy'].unique()

array(['LEP', 'Conditional'], dtype=object)

## Load and process CB data

In [76]:
# cb_raw_df = pd.read_csv(os.path.join(INPUT_DATA_DIR_PATH, "costs_benefits_sisepuede_results_sisepuede_run_2026-01-29T15;28;40.322709_tornado_raw.csv"))
cb_raw_df = pd.read_csv(os.path.join(INPUT_DATA_DIR_PATH, "cb_sisepuede_results_run_sisepuede_run_2026-03-13T07;36;11.592365.csv"))
cb_raw_df.head()

,strategy_code,future_id,region,time_period,difference_variable,variable_value_baseline,variable_value_pathway,difference_value,variable,value,...,sector,cb_type,item_1,item_2,Year,strategy,strategy_id,primary_id,ids,gdp_mmm_usd
0,BASE,0.0,libya,8.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,...,agrc,crop_value,crops_produced,bevs_and_spices,2023.0,BASE,0,0,cb:agrc:crop_value:crops_produced:bevs_and_spi...,101.162039
1,BASE,0.0,libya,9.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,...,agrc,crop_value,crops_produced,bevs_and_spices,2024.0,BASE,0,0,cb:agrc:crop_value:crops_produced:bevs_and_spi...,105.576742
2,BASE,0.0,libya,10.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,...,agrc,crop_value,crops_produced,bevs_and_spices,2025.0,BASE,0,0,cb:agrc:crop_value:crops_produced:bevs_and_spi...,107.688276
3,BASE,0.0,libya,11.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,...,agrc,crop_value,crops_produced,bevs_and_spices,2026.0,BASE,0,0,cb:agrc:crop_value:crops_produced:bevs_and_spi...,109.842042
4,BASE,0.0,libya,12.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,...,agrc,crop_value,crops_produced,bevs_and_spices,2027.0,BASE,0,0,cb:agrc:crop_value:crops_produced:bevs_and_spi...,112.038883


In [77]:
# --- Create a copy of the raw data ---
cb_data = cb_raw_df.copy()

cb_data.head()

,strategy_code,future_id,region,time_period,difference_variable,variable_value_baseline,variable_value_pathway,difference_value,variable,value,...,sector,cb_type,item_1,item_2,Year,strategy,strategy_id,primary_id,ids,gdp_mmm_usd
0,BASE,0.0,libya,8.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,...,agrc,crop_value,crops_produced,bevs_and_spices,2023.0,BASE,0,0,cb:agrc:crop_value:crops_produced:bevs_and_spi...,101.162039
1,BASE,0.0,libya,9.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,...,agrc,crop_value,crops_produced,bevs_and_spices,2024.0,BASE,0,0,cb:agrc:crop_value:crops_produced:bevs_and_spi...,105.576742
2,BASE,0.0,libya,10.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,...,agrc,crop_value,crops_produced,bevs_and_spices,2025.0,BASE,0,0,cb:agrc:crop_value:crops_produced:bevs_and_spi...,107.688276
3,BASE,0.0,libya,11.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,...,agrc,crop_value,crops_produced,bevs_and_spices,2026.0,BASE,0,0,cb:agrc:crop_value:crops_produced:bevs_and_spi...,109.842042
4,BASE,0.0,libya,12.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,...,agrc,crop_value,crops_produced,bevs_and_spices,2027.0,BASE,0,0,cb:agrc:crop_value:crops_produced:bevs_and_spi...,112.038883


In [78]:
cb_data["sector"].unique()

array(['agrc', 'ccsq', 'entc', 'inen', 'ippu', 'lndu', 'lsmm', 'lvst',
       'scoe', 'soil', 'trns', 'trww', 'wali', 'waso', 'fgtv'],
      dtype=object)

In [79]:
cb_data.head()

,strategy_code,future_id,region,time_period,difference_variable,variable_value_baseline,variable_value_pathway,difference_value,variable,value,...,sector,cb_type,item_1,item_2,Year,strategy,strategy_id,primary_id,ids,gdp_mmm_usd
0,BASE,0.0,libya,8.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,...,agrc,crop_value,crops_produced,bevs_and_spices,2023.0,BASE,0,0,cb:agrc:crop_value:crops_produced:bevs_and_spi...,101.162039
1,BASE,0.0,libya,9.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,...,agrc,crop_value,crops_produced,bevs_and_spices,2024.0,BASE,0,0,cb:agrc:crop_value:crops_produced:bevs_and_spi...,105.576742
2,BASE,0.0,libya,10.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,...,agrc,crop_value,crops_produced,bevs_and_spices,2025.0,BASE,0,0,cb:agrc:crop_value:crops_produced:bevs_and_spi...,107.688276
3,BASE,0.0,libya,11.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,...,agrc,crop_value,crops_produced,bevs_and_spices,2026.0,BASE,0,0,cb:agrc:crop_value:crops_produced:bevs_and_spi...,109.842042
4,BASE,0.0,libya,12.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,...,agrc,crop_value,crops_produced,bevs_and_spices,2027.0,BASE,0,0,cb:agrc:crop_value:crops_produced:bevs_and_spi...,112.038883


In [80]:
cb_data['cb_type'].unique()

array(['crop_value', 'sector_specific', 'air_pollution', 'technical_cost',
       'ippu_value', 'ecosystem_services', 'fuel_cost', 'lvst_value',
       'land_pollution', 'congestion', 'road_safety', 'system_cost',
       'water_pollution', 'human_health', 'env_pollution',
       'technical_savings', 'consumer_savings'], dtype=object)

In [81]:
cb_data = cb_data[cb_data['cb_type']=='technical_cost']

In [82]:
cb_data['cb_type'].unique()

array(['technical_cost'], dtype=object)

In [83]:
cb_data['strategy_code'].unique()

array(['BASE', 'PFLO:LEP', 'PFLO:CONDITIONAL'], dtype=object)

In [84]:
cb_data['strategy_code'].unique()

array(['BASE', 'PFLO:LEP', 'PFLO:CONDITIONAL'], dtype=object)

In [85]:
cb_data = cb_data[cb_data['strategy_code']!='BASE']

In [86]:
cb_data['strategy_code'].unique()

array(['PFLO:LEP', 'PFLO:CONDITIONAL'], dtype=object)

In [87]:
tornado_emissions_agg_df['subsector'].unique()

array(['1.A.1 - Energy Industries',
       '1.A.2 - Manufacturing Industries and Construction',
       '1.A.3 - Transport', '1.A.4 - Other Sectors',
       '1.B - Fugitive emissions from fuels', '2 - IPPU',
       '3.A - Livestock',
       '3.C - Aggregate sources and non-CO2 emissions sources on land',
       '4.A - Solid Waste Disposal',
       '4.D - Wastewater Treatment and Discharge', '5- CCSQ'],
      dtype=object)

In [88]:
SECTOR_TO_SUBSECTOR = {
    
    "entc": "1.A.1 - Energy Industries",
    "inen": "1.A.2 - Manufacturing Industries and Construction",
    "trns": "1.A.3 - Transport",
    "scoe": "1.A.4 - Other Sectors",

    'ippu': '2 - IPPU',

    "fgtv": "1.B - Fugitive emissions from fuels",

    "lvst": "3.A - Livestock",
    "lsmm": "3.A - Livestock",

    "soil": "3.C - Aggregate sources and non-CO2 emissions sources on land",
    "agr": "3.C - Aggregate sources and non-CO2 emissions sources on land",

    "waso": "4.A - Solid Waste Disposal",
    "trww": "4.D - Wastewater Treatment and Discharge",

    "ccsq": "5- CCSQ",

}

def add_subsector(df: pd.DataFrame, sector_col: str = "sector") -> pd.DataFrame:
    df = df.copy()
    df["subsector_ssp"] = df[sector_col].str.lower().map(SECTOR_TO_SUBSECTOR)
    return df

cb_data = add_subsector(cb_data)

In [89]:
# aggregate sum(value) grouped by strategy_code and sector
cb_data = (
    cb_data.groupby(["strategy_code", "strategy_id", "primary_id", "subsector_ssp"], as_index=False)["value"]
      .sum()
      .rename(columns={"value": "cumulative"})
)
cb_data.head(20)

,strategy_code,strategy_id,primary_id,subsector_ssp,cumulative
0,PFLO:CONDITIONAL,6005,76076,1.A.1 - Energy Industries,-45.481827
1,PFLO:CONDITIONAL,6005,76076,1.A.2 - Manufacturing Industries and Construction,-15.796292
2,PFLO:CONDITIONAL,6005,76076,1.A.3 - Transport,-4.444676
3,PFLO:CONDITIONAL,6005,76076,1.A.4 - Other Sectors,-5.450558
4,PFLO:CONDITIONAL,6005,76076,1.B - Fugitive emissions from fuels,-0.201176
5,PFLO:CONDITIONAL,6005,76076,2 - IPPU,-0.365549
6,PFLO:CONDITIONAL,6005,76076,3.A - Livestock,-0.083430
7,PFLO:CONDITIONAL,6005,76076,3.C - Aggregate sources and non-CO2 emissions ...,0.023040
8,PFLO:CONDITIONAL,6005,76076,4.A - Solid Waste Disposal,0.763134
9,PFLO:CONDITIONAL,6005,76076,4.D - Wastewater Treatment and Discharge,-1.625015


In [90]:
tornado_emissions_agg_df.head()

,strategy_id,primary_id,strategy,sector,subsector,emission_total,base_emission_total,emission_diff
11,6004.0,75075.0,LEP,1 - Energy,1.A.1 - Energy Industries,719.263397,960.395602,-241.1
12,6004.0,75075.0,LEP,1 - Energy,1.A.2 - Manufacturing Industries and Construction,32.437106,51.611824,-19.2
13,6004.0,75075.0,LEP,1 - Energy,1.A.3 - Transport,712.674649,873.965289,-161.3
14,6004.0,75075.0,LEP,1 - Energy,1.A.4 - Other Sectors,20.739560,31.779867,-11.0
15,6004.0,75075.0,LEP,1 - Energy,1.B - Fugitive emissions from fuels,933.572988,793.721312,139.9


In [91]:
merged_df = tornado_emissions_agg_df.merge(
    cb_data[["strategy_id", "primary_id", "subsector_ssp", "cumulative"]],
    left_on=["strategy_id", "primary_id", "subsector"],
    right_on=["strategy_id", "primary_id", "subsector_ssp"],
    how="left"
).drop(columns="subsector_ssp")

merged_df.head(20)
  

,strategy_id,primary_id,strategy,sector,subsector,emission_total,base_emission_total,emission_diff,cumulative
0,6004.0,75075.0,LEP,1 - Energy,1.A.1 - Energy Industries,719.263397,960.395602,-241.1,-7.466506
1,6004.0,75075.0,LEP,1 - Energy,1.A.2 - Manufacturing Industries and Construction,32.437106,51.611824,-19.2,-7.955268
2,6004.0,75075.0,LEP,1 - Energy,1.A.3 - Transport,712.674649,873.965289,-161.3,-1.259032
3,6004.0,75075.0,LEP,1 - Energy,1.A.4 - Other Sectors,20.739560,31.779867,-11.0,-0.708690
4,6004.0,75075.0,LEP,1 - Energy,1.B - Fugitive emissions from fuels,933.572988,793.721312,139.9,1.348553
5,6004.0,75075.0,LEP,2 - Industrial Processes and Product Use,2 - IPPU,87.849474,122.231498,-34.4,0.000000
6,6004.0,75075.0,LEP,"3 - Agriculture, Forestry, and Other Land Use",3.A - Livestock,32.903191,32.904753,-0.0,-0.052938
7,6004.0,75075.0,LEP,"3 - Agriculture, Forestry, and Other Land Use",3.C - Aggregate sources and non-CO2 emissions ...,23.021140,23.129455,-0.1,0.004589
8,6004.0,75075.0,LEP,4 - Waste,4.A - Solid Waste Disposal,14.642588,14.689550,-0.0,0.265594
9,6004.0,75075.0,LEP,4 - Waste,4.D - Wastewater Treatment and Discharge,13.544044,13.686307,-0.1,-0.325003


In [92]:
OUTPUT_DATA_DIR_PATH

'/Users/fabianfuentes/git/ssp_libya/ssp_modeling/cost-benefits/tornado_plot/data/output'

In [93]:
merged_df.to_csv(os.path.join(OUTPUT_DATA_DIR_PATH, "mac_sector_libya.csv"), index=False)